In [ ]:
pip install --user langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [39]:
from langchain_core.documents import Document

In [ ]:
pip install --user langchain_text_splitters

In [40]:
sample_doc = Document(
    page_content="Hello world",
    metadata= {"source":"https://www.google.com"}
)

In [41]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello world')

In [42]:
type(sample_doc)

langchain_core.documents.base.Document

In [43]:
#Text Data
from langchain_community.document_loaders.text import TextLoader

In [44]:
loader=TextLoader("python.txt",encoding="utf-8")

In [45]:
document = loader.load()

In [46]:
document

[Document(metadata={'source': 'python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and more.\n

In [10]:
# #PDF Data
# from langchain_community.document_loaders.pdf import PyPDFLoader
# pdf_loader = PyPDFLoader("./pdfs/research1.pdf")
# doc = pdf_loader.load()
# # doc

In [11]:
# import sys
# print(sys.version)

## Ingestion Pipeline

In [47]:
#Data => Documents
import os 
from langchain_community.document_loaders.pdf import PyPDFLoader

In [48]:
def load_all_pdfs():
    folder_path = "pdfs"
    num_docs = 0
    all_docs=[]

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            #complete path
            pdf_path = os.path.join(folder_path,filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs+=1
    print(num_docs)
    print(len(all_docs))
    return all_docs

In [49]:
all_pdfs_doc = load_all_pdfs()

2
32


In [50]:
type(all_pdfs_doc[0])

langchain_core.documents.base.Document

In [51]:
#Chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter 
# creates chunks

In [52]:
def split_doc(documents,chunk_size=500,chunk_overlap=50):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunked_doc = text_splitter.split_documents(documents)
    return chunked_doc

In [53]:
chunks = split_doc(all_pdfs_doc)

In [54]:
len(chunks)

321

In [38]:
pip show  sentence-transformers

Name: sentence-transformers
Version: 5.4.1
Summary: Embeddings, Retrieval, and Reranking
Home-page: https://www.SBERT.net
Author: 
Author-email: Nils Reimers <info@nils-reimers.de>, Tom Aarsen <tom.aarsen@huggingface.co>
License: Apache 2.0
Location: C:\Users\eppah\AppData\Roaming\Python\Python311\site-packages
Requires: huggingface-hub, numpy, scikit-learn, scipy, torch, tqdm, transformers, typing_extensions
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [20]:
# sentence transformer - to convert doduments into the embedding 

In [55]:
from sentence_transformers import SentenceTransformer

In [56]:
class EmbeddingManager:
    def __init__(self,model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        print("loading model....",self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=",self.model.get_sentence_embedding_dimension())
    def generate_embeddings(self,text):
        embeddings=self.model.encode(text,show_progress_bar = True)
        print("Embeddings shape:",embeddings.shape)
        return embeddings
        

In [57]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 3270.25it/s]


embedding dimensions= 384


C:\Users\eppah\AppData\Local\Temp\ipykernel_17816\2071319601.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=",self.model.get_sentence_embedding_dimension())


## Vector Store

In [118]:
# pip install --user uuid

In [58]:
import chromadb
import uuid

In [91]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        
        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embeddings_list
        )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())
    def delete_collection(self):
        self.client.delete_collection(name=self.collection_name)
        print("collection deleted")

In [92]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 0


In [90]:
# vector_store.delete_collection()

collection deleted


In [93]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)

Batches: 100%|█████████████████████████████████████████████████████████████████████████| 11/11 [00:36<00:00,  3.36s/it]


Embeddings shape: (321, 384)
total documents added in vector store= 321
docs in collection: 321


In [110]:
# data = vector_store.collection.get()
# print(data)

In [107]:
# data = vector_store.collection.get(
#     include=["embeddings"]
# )

# print(data["embeddings"])

In [62]:
from sklearn.metrics.pairwise import cosine_similarity

In [111]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [112]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [113]:
rag_retriever.retrieve("What is encoder decoder")

Batches: 100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 25.00it/s]

Embeddings shape: (1, 384)
retrieved 4 documents


[{'id': 'doc_2ba0f7ae-d556-4f60-b2fc-cf409d4ae401',
  'document': 'positional encodings in both the encoder and decoder stacks. For the base model, we use a rate of\nPdrop = 0.1.\n7\neppaharsha13@gmail.com',
  'metadata': {'editors': 'I. Guyon and U.V. Luxburg and S. Bengio and H. Wallach and R. Fergus and S. Vishwanathan and R. Garnett',
   'book': 'Advances in Neural Information Processing Systems 30',
   'page': 6,
   'page_label': '7',
   'doc_index': 294,
   'total_pages': 11,
   'source': 'pdfs\\research2.pdf',
   'created': '2017',
   'producer': 'pdfcpu v0.9.1 dev',
   'type': 'Conference Proceedings',
   'title': 'Attention is All you Need',
   'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely ona

## Integrate with LLMs

## OpenAI-GPT

In [119]:
API_KEY_OPENAI = "paste your api key"

In [68]:
# pip install --user langchain-openai

In [114]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key=API_KEY_OPENAI,
    model="gpt-5.4",
    temperature=0.1,
    max_tokens=1024
)

In [115]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke(prompt) # expecting a string as prompt
    return response.content

In [116]:
answer = generate_output("what is RAG?", rag_retriever, llm)

Batches: 100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 23.81it/s]


Embeddings shape: (1, 384)
retrieved 3 documents


In [117]:
print(answer)

RAG stands for **Retrieval-Augmented Generation**.

It is a framework that combines:

- **Retrieval**: finding relevant external information from sources like documents, databases, or knowledge bases
- **Generation**: using a language model to produce an answer based on the retrieved information

In simple terms, instead of relying only on what a language model memorized during training, RAG first **looks up useful knowledge** and then **uses that knowledge to generate a response**.

From the context, the survey describes RAG as an evolving field integrated with **LLMs**, with paradigms such as **naive RAG**, more advanced methods, and future directions. Its goal is to improve the effectiveness of language models by grounding responses in retrieved evidence.

A short definition:
> **RAG is a method for enhancing large language models by retrieving relevant information and using it to generate more accurate, informed answers.**
